In [9]:
!pip install python-dotenv

In [3]:
!pip install tuya-connector-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 6.8 MB/s eta 0:00:0000:0100:01


In [4]:
from dotenv import load_dotenv
import os
import hashlib
import random
import string
import time
import requests

def calculate_sha512(text):
    return hashlib.sha512(text.encode('utf-8')).hexdigest()

def add_authorization_parameters(method, parameters, key, secret):
    parameters["apiKey"] = key
    parameters["time"] = str(int(time.time()))

    rand = ''.join(random.choices(string.ascii_lowercase, k=6))
    
    sorted_params = sorted(parameters.items())
    query_string = '&'.join(f'{k}={v}' for k, v in sorted_params)
    
    api_sig_base = f"{rand}/{method}?{query_string}#{secret}"
    api_sig = rand + calculate_sha512(api_sig_base)
    
    parameters["apiSig"] = api_sig

# Load environment variables from a specified .env file
load_dotenv(dotenv_path='/home/jovyan/work/.env')

def codeforces_api_request(method, parameters):
    base_url = "https://codeforces.com/api/"
    url = f"{base_url}{method}"

    key = os.getenv("CODEFORCES_API_KEY")
    secret = os.getenv("CODEFORCES_API_SECRET")
    
    if not key or not secret:
        raise ValueError("API key and secret must be set as environment variables.")
    
    add_authorization_parameters(method, parameters, key, secret)
    
    combined_parameters = '&'.join(f'{k}={v}' for k, v in parameters.items())
    full_url = f"{url}?{combined_parameters}"
    # # print(f"Parameters before :: {parameters}")  # For debugging
    # # print(f"New Combined Parameters are :: {combined_parameters}")  # For debugging
    # # print(f"Formed Request is as follows :: {full_url}")  # For debugging
    response = requests.get(full_url)
    if response.status_code == 200:
        return response.json()
    else:
        print("Failed to fetch data")
        return None
def map_rating_to_color(rating):
    # Map Codeforces rating to color
    if rating < 1200:
        return "blue"
    elif rating < 1400:
        return "green"
    elif rating < 1600:
        return "cyan"
    elif rating < 1900:
        return "yellow"
    elif rating < 2100:
        return "orange"
    else:
        return "red"

def main():
    method = "user.rating"
    parameters = {"handle": "hanisntsolo"}
    data = codeforces_api_request(method, parameters)
    # print(data)
    handle = "hanisntsolo"
    method = "user.info"
    parameters = {"handles": handle}
    data = codeforces_api_request(method, parameters)
    print(data)
    
if __name__ == "__main__":
    main()


{'status': 'OK', 'result': [{'lastName': 'Singh', 'country': 'India', 'lastOnlineTimeSeconds': 1720069030, 'city': 'Lucknow', 'rating': 670, 'friendOfCount': 0, 'titlePhoto': 'https://userpic.codeforces.org/2151510/title/e11136949f267060.jpg', 'handle': 'hanisntsolo', 'avatar': 'https://userpic.codeforces.org/2151510/avatar/bfaa2d6fb85d9935.jpg', 'firstName': 'Dhirendra', 'contribution': 0, 'organization': 'Citi', 'rank': 'newbie', 'maxRating': 863, 'registrationTimeSeconds': 1628178692, 'maxRank': 'newbie'}]}
